# GPT-2 for Time Series Forecasting

## What We'll Learn

Transformer models like **GPT-2** excel at sequential prediction because they learn patterns in ordered data. While designed for text, the core mechanism—**self-attention** over sequences—applies equally to time series.

In this notebook, we'll:
- Understand why GPT-2's architecture suits time series forecasting
- Learn how to **discretize** continuous time series values into tokens
- Adapt GPT-2 to predict future time series values
- Compare approaches: discretization vs. direct regression
- Visualize attention patterns over time sequences

**Key Insight**: Transformers don't "know" about text vs. numbers. They process sequences of embeddings. If we can map time series to embeddings, GPT-2 becomes a powerful forecasting model.

## 1. The Core Idea: Text vs. Time Series

| **Text Modeling** | **Time Series Modeling** |
|-------------------|-------------------------|
| Input: sequence of tokens | Input: sequence of numerical values |
| Predict: next token | Predict: next value(s) |
| Tokens: discrete (vocab) | Values: continuous (real numbers) |
| Embedding: lookup table | Embedding: learned projection or binning |

**Two approaches**:
1. **Discretization**: Bin time series values into discrete "tokens" → use standard GPT-2
2. **Regression Head**: Keep values continuous → replace GPT-2's token prediction head with a regression layer

We'll explore **discretization** first as it requires minimal changes to GPT-2.

## 2. Setup

### Configuration

All hyperparameters in one place for easy experimentation.

In [ ]:
CONFIG = {
    # Reproducibility
    'seed': 42,  # Random seed
    
    # Data Generation
    'num_sequences': 1000,  # Number of time series sequences to generate
    'sequence_length': 128,  # Length of each sequence
    'context_length': 64,  # How many past values to condition on
    'prediction_length': 1,  # How many future values to predict
    
    # Time Series Discretization
    'num_bins': 256,  # Number of discrete bins for values (like vocab size)
    'value_range': (-3.0, 3.0),  # Expected range of normalized values
    
    # Model Architecture (GPT-2 small)
    'n_embd': 128,  # Embedding dimension (reduced from 768 for faster training)
    'n_layer': 2,  # Number of transformer blocks (reduced from 12)
    'n_head': 4,  # Number of attention heads (reduced from 12)
    'dropout': 0.1,  # Dropout rate
    
    # Training
    'batch_size': 128,  # Batch size
    'learning_rate': 3e-4,  # Learning rate
    'max_epochs': 5,  # Maximum training epochs
    'early_stop_patience': 3,  # Early stopping patience
    'num_workers': 0,  # DataLoader workers
}

### Random Seed and Device

Set up reproducibility and automatic device detection.

In [ ]:
from aiml_notebooks import get_device, set_seed

%load_ext autoreload
%autoreload 2

set_seed(CONFIG['seed'])
device = get_device()
print(f"Using device: {device}")

## 3. Generate Synthetic Time Series Data

### Understanding the Data

We'll generate synthetic time series with **multiple patterns**:
- **Sine waves**: Smooth periodic oscillations
- **Trends**: Linear increasing/decreasing patterns
- **Noise**: Random fluctuations
- **Seasonality**: Multiple frequencies combined

This mix creates realistic complexity while remaining predictable enough to learn.

In [ ]:
import numpy as np
import torch

def generate_synthetic_time_series(num_sequences, sequence_length, seed=None):
    """
    Generate synthetic time series with multiple patterns:
    - Sine waves with random frequency and phase
    - Linear trends
    - Gaussian noise
    """
    if seed is not None:
        np.random.seed(seed)
    
    sequences = []
    
    for _ in range(num_sequences):
        t = np.linspace(0, 4 * np.pi, sequence_length)
        
        # Random frequency and phase for sine wave
        freq = np.random.uniform(0.5, 2.0)
        phase = np.random.uniform(0, 2 * np.pi)
        sine = np.sin(freq * t + phase)
        
        # Random trend
        trend = np.random.uniform(-0.5, 0.5) * np.linspace(-1, 1, sequence_length)
        
        # Noise
        noise = np.random.normal(0, 0.1, sequence_length)
        
        # Combine components
        series = sine + trend + noise
        sequences.append(series)
    
    return np.array(sequences, dtype=np.float32)

# Generate data
all_sequences = generate_synthetic_time_series(
    CONFIG['num_sequences'],
    CONFIG['sequence_length'],
    seed=CONFIG['seed']
)

print(f"Generated shape: {all_sequences.shape}")  # (num_sequences, sequence_length)
print(f"Value range: [{all_sequences.min():.2f}, {all_sequences.max():.2f}]")

### Visualize Sample Time Series

Let's look at a few examples to understand the patterns.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(14, 8))

for i, ax in enumerate(axes):
    ax.plot(all_sequences[i], linewidth=2, color='#4ECDC4')
    ax.set_title(f'Time Series {i+1}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time Step', fontsize=10)
    ax.set_ylabel('Value', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Each series combines sine waves, trends, and noise.")

## 4. Discretization: From Continuous to Tokens

### Why Discretize?

GPT-2 expects **discrete tokens** (integers 0 to vocab_size-1). We need to convert continuous values to this format.

**Approach**: **Uniform binning**
- Divide the value range into `num_bins` equal intervals
- Map each value to its bin index
- Example: value=-1.5 in range [-3, 3] with 256 bins → bin ~42

**Trade-off**: More bins = better precision but larger vocab and potentially harder learning.

In [ ]:
class TimeSeriesDiscretizer:
    """Convert continuous time series values to/from discrete bins."""
    
    def __init__(self, num_bins, value_range):
        self.num_bins = num_bins
        self.value_min, self.value_max = value_range
        self.bin_width = (self.value_max - self.value_min) / num_bins
    
    def encode(self, values):
        """Convert continuous values to bin indices."""
        # Clip to range and normalize to [0, 1]
        clipped = np.clip(values, self.value_min, self.value_max)
        normalized = (clipped - self.value_min) / (self.value_max - self.value_min)
        
        # Convert to bin indices [0, num_bins-1]
        bins = (normalized * self.num_bins).astype(np.int64)
        bins = np.clip(bins, 0, self.num_bins - 1)
        
        return bins
    
    def decode(self, bins):
        """Convert bin indices back to continuous values (using bin centers)."""
        # Use center of each bin
        normalized = (bins + 0.5) / self.num_bins
        values = normalized * (self.value_max - self.value_min) + self.value_min
        return values

# Create discretizer
discretizer = TimeSeriesDiscretizer(
    CONFIG['num_bins'],
    CONFIG['value_range']
)

# Test on a sample
sample_values = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
sample_bins = discretizer.encode(sample_values)
reconstructed = discretizer.decode(sample_bins)

print("Discretization test:")
print(f"  Original:      {sample_values}")
print(f"  Bins:          {sample_bins}")
print(f"  Reconstructed: {reconstructed}")
print(f"  Error:         {np.abs(sample_values - reconstructed).mean():.4f}")

### Visualize Discretization Effect

Let's see how discretization affects a continuous signal.

In [ ]:
# Take first sequence
original = all_sequences[0]
bins = discretizer.encode(original)
reconstructed = discretizer.decode(bins)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Original vs reconstructed
ax1.plot(original, label='Original (Continuous)', linewidth=2, color='#4ECDC4', alpha=0.7)
ax1.plot(reconstructed, label='Reconstructed (Discretized)', linewidth=2, 
         color='#FF6B6B', linestyle='--', alpha=0.7)
ax1.set_title('Discretization Effect on Time Series', fontsize=14, fontweight='bold')
ax1.set_xlabel('Time Step', fontsize=12)
ax1.set_ylabel('Value', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Reconstruction error
error = np.abs(original - reconstructed)
ax2.plot(error, linewidth=2, color='#95E1D3')
ax2.set_title('Reconstruction Error (Quantization Loss)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Time Step', fontsize=12)
ax2.set_ylabel('Absolute Error', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean reconstruction error: {error.mean():.4f}")
print(f"Max reconstruction error: {error.max():.4f}")

## 5. Create Dataset for Forecasting

### Sliding Window Approach

For each sequence, we create multiple training examples using a **sliding window**:
- **Input**: `context_length` past values (e.g., last 64 time steps)
- **Target**: Next value(s) to predict

This is analogous to language modeling where we predict the next token given previous tokens.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TimeSeriesDataset(Dataset):
    """Dataset for time series forecasting using sliding windows."""
    
    def __init__(self, sequences, context_length, discretizer):
        self.sequences = sequences
        self.context_length = context_length
        self.discretizer = discretizer
        
        # Discretize all sequences upfront
        self.discretized = discretizer.encode(sequences)
    
    def __len__(self):
        # Each sequence generates (seq_len - context_length) examples
        return len(self.sequences) * (self.sequences.shape[1] - self.context_length)
    
    def __getitem__(self, idx):
        # Map flat index to (sequence_idx, position)
        seq_len = self.sequences.shape[1]
        num_windows = seq_len - self.context_length
        
        seq_idx = idx // num_windows
        pos = idx % num_windows
        
        # Extract window: [pos : pos+context_length] predicts [pos+context_length]
        context = self.discretized[seq_idx, pos:pos + self.context_length]
        target = self.discretized[seq_idx, pos + self.context_length]
        
        return torch.tensor(context, dtype=torch.long), torch.tensor(target, dtype=torch.long)

# Split data
train_size = int(0.8 * len(all_sequences))
val_size = int(0.1 * len(all_sequences))

train_sequences = all_sequences[:train_size]
val_sequences = all_sequences[train_size:train_size + val_size]
test_sequences = all_sequences[train_size + val_size:]

# Create datasets
train_dataset = TimeSeriesDataset(train_sequences, CONFIG['context_length'], discretizer)
val_dataset = TimeSeriesDataset(val_sequences, CONFIG['context_length'], discretizer)
test_dataset = TimeSeriesDataset(test_sequences, CONFIG['context_length'], discretizer)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers']
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers']
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers']
)

print(f"Train windows: {len(train_dataset):,}")
print(f"Val windows:   {len(val_dataset):,}")
print(f"Test windows:  {len(test_dataset):,}")

# Check a batch
sample_context, sample_target = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  Context: {sample_context.shape}  (batch_size, context_length)")
print(f"  Target:  {sample_target.shape}  (batch_size,)")

## 6. Implement GPT-2 Architecture

### Core Components

We'll implement a **simplified GPT-2** with:
1. **Token Embeddings**: Map bin indices to dense vectors
2. **Positional Embeddings**: Encode position in sequence
3. **Transformer Blocks**: Multi-head self-attention + feedforward layers
4. **Output Head**: Project to vocabulary (bin) logits

Key difference from text: Our "vocabulary" is `num_bins` instead of 50k words.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math

class CausalSelfAttention(nn.Module):
    """Multi-head masked self-attention with causal mask."""
    
    def __init__(self, n_embd, n_head, context_length, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head
        
        # Q, K, V projections for all heads (batched)
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        
        # Causal mask to ensure we only attend to past
        self.register_buffer(
            "bias",
            torch.tril(torch.ones(context_length, context_length))
            .view(1, 1, context_length, context_length)
        )
    
    def forward(self, x):
        B, T, C = x.size()  # batch, sequence length, embedding dim
        
        # Calculate Q, K, V
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        
        # Reshape for multi-head attention: (B, T, C) -> (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        
        # Attention scores
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        
        # Apply causal mask
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        
        # Apply attention to values
        y = att @ v  # (B, n_head, T, head_dim)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        
        # Output projection
        y = self.resid_dropout(self.c_proj(y))
        
        return y


class MLP(nn.Module):
    """Feedforward network."""
    
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
        self.gelu = nn.GELU()
    
    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


class TransformerBlock(nn.Module):
    """Transformer block: attention + MLP with residuals and layer norms."""
    
    def __init__(self, n_embd, n_head, context_length, dropout):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, context_length, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)
    
    def forward(self, x):
        # Pre-norm architecture (norm before attention/MLP)
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


print("GPT-2 components implemented: CausalSelfAttention, MLP, TransformerBlock")

### Complete GPT-2 Model

Now we assemble the full model using PyTorch Lightning.

In [ ]:
import lightning as L

class GPT2TimeSeries(L.LightningModule):
    """GPT-2 adapted for time series forecasting."""
    
    def __init__(
        self,
        vocab_size=CONFIG['num_bins'],
        context_length=CONFIG['context_length'],
        n_embd=CONFIG['n_embd'],
        n_layer=CONFIG['n_layer'],
        n_head=CONFIG['n_head'],
        dropout=CONFIG['dropout'],
        learning_rate=CONFIG['learning_rate']
    ):
        super().__init__()
        self.save_hyperparameters()
        
        # Token + position embeddings
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(context_length, n_embd)
        self.drop = nn.Dropout(dropout)
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(n_embd, n_head, context_length, dropout)
            for _ in range(n_layer)
        ])
        
        # Output layer
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        
        # Loss
        self.criterion = nn.CrossEntropyLoss()
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx):
        B, T = idx.size()
        
        # Token embeddings
        tok_emb = self.token_embedding(idx)  # (B, T, n_embd)
        
        # Position embeddings
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)  # (T,)
        pos_emb = self.position_embedding(pos)  # (T, n_embd)
        
        # Combine and apply dropout
        x = self.drop(tok_emb + pos_emb)
        
        # Transformer blocks
        for block in self.blocks:
            x = block(x)
        
        # Final layer norm and projection
        x = self.ln_f(x)
        logits = self.head(x)  # (B, T, vocab_size)
        
        return logits
    
    def training_step(self, batch, batch_idx):
        context, target = batch
        
        # Forward pass - use only last position for prediction
        logits = self(context)  # (B, T, vocab_size)
        logits = logits[:, -1, :]  # (B, vocab_size) - last time step
        
        # Loss
        loss = self.criterion(logits, target)
        
        # Accuracy
        preds = logits.argmax(dim=-1)
        acc = (preds == target).float().mean()
        
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc', acc, prog_bar=True)
        
        return loss
    
    def validation_step(self, batch, batch_idx):
        context, target = batch
        
        logits = self(context)
        logits = logits[:, -1, :]
        
        loss = self.criterion(logits, target)
        preds = logits.argmax(dim=-1)
        acc = (preds == target).float().mean()
        
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)
        
        return loss
    
    def configure_optimizers(self):
        # Use AdamW with weight decay (as in GPT-2)
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate, weight_decay=0.01)


# Create model
model = GPT2TimeSeries()

from aiml_notebooks import count_parameters
print(f"\nModel has {count_parameters(model):,} trainable parameters")

## 7. Train the Model

We'll use PyTorch Lightning's trainer with early stopping to prevent overfitting.

In [ ]:
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import EarlyStopping

# Setup logger
logger = CSVLogger('logs', name='gpt2_time_series')

# Early stopping callback
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=CONFIG['early_stop_patience'],
    mode='min',
    verbose=False
)

# Create trainer
trainer = L.Trainer(
    max_epochs=CONFIG['max_epochs'],
    accelerator='auto',
    devices=1,
    logger=logger,
    callbacks=[early_stop],
    enable_progress_bar=True,
    gradient_clip_val=1.0  # Clip gradients for stability
)

# Train
print("Starting training...\n")
trainer.fit(model, train_loader, val_loader)

### Plot Training Curves

Visualize how the model learned over time.

In [ ]:
import pandas as pd

# Read metrics
metrics = pd.read_csv(f'{logger.log_dir}/metrics.csv')

# Aggregate by epoch
train_metrics = metrics[['epoch', 'train_loss', 'train_acc']].dropna()
val_metrics = metrics[['epoch', 'val_loss', 'val_acc']].dropna()
train_metrics = train_metrics.groupby('epoch').mean().reset_index()
val_metrics = val_metrics.groupby('epoch').mean().reset_index()

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(train_metrics['epoch'], train_metrics['train_loss'],
         label='Train', marker='o', linewidth=2, color='#4ECDC4')
ax1.plot(val_metrics['epoch'], val_metrics['val_loss'],
         label='Validation', marker='s', linewidth=2, color='#FF6B6B')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

if trainer.early_stopping_callback and trainer.early_stopping_callback.stopped_epoch > 0:
    stop_epoch = trainer.early_stopping_callback.stopped_epoch
    ax1.axvline(x=stop_epoch, color='red', linestyle='--', alpha=0.5, label='Early Stop')

# Accuracy curves
ax2.plot(train_metrics['epoch'], train_metrics['train_acc'] * 100,
         label='Train', marker='o', linewidth=2, color='#4ECDC4')
ax2.plot(val_metrics['epoch'], val_metrics['val_acc'] * 100,
         label='Validation', marker='s', linewidth=2, color='#FF6B6B')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

if trainer.early_stopping_callback and trainer.early_stopping_callback.stopped_epoch > 0:
    ax2.axvline(x=stop_epoch, color='red', linestyle='--', alpha=0.5, label='Early Stop')

plt.tight_layout()
plt.show()

# Statistics
epochs_trained = len(train_metrics)
print(f"\nTraining Statistics:")
print(f"  Epochs trained: {epochs_trained} / {CONFIG['max_epochs']}")
if trainer.early_stopping_callback and trainer.early_stopping_callback.stopped_epoch > 0:
    print(f"  Early stopping triggered at epoch {stop_epoch}")
print(f"\nFinal Results:")
print(f"  Train Accuracy: {train_metrics['train_acc'].iloc[-1]*100:.2f}%")
print(f"  Val Accuracy: {val_metrics['val_acc'].iloc[-1]*100:.2f}%")

## 8. Evaluate Forecasting Performance

### Single-Step Prediction

Let's test how well the model predicts the next value given a context window.

In [ ]:
model.eval()
model = model.to(device)  # Ensure model is on correct device

# Get a test sequence
test_idx = 0
test_sequence = test_sequences[test_idx]

# Use first context_length values to predict the rest
context = test_sequence[:CONFIG['context_length']]
true_future = test_sequence[CONFIG['context_length']:]

# Make predictions one step at a time
predictions = []
current_context = discretizer.encode(context).copy()

with torch.no_grad():
    for _ in range(len(true_future)):
        # Prepare input
        context_tensor = torch.tensor(current_context, dtype=torch.long).unsqueeze(0).to(device)
        
        # Predict next value
        logits = model(context_tensor)
        next_token = logits[0, -1, :].argmax().item()
        
        predictions.append(next_token)
        
        # Update context (sliding window)
        current_context = np.append(current_context[1:], next_token)

# Convert predictions back to continuous values
predicted_values = discretizer.decode(np.array(predictions))

# Plot
fig, ax = plt.subplots(figsize=(14, 6))

full_time = np.arange(len(test_sequence))
context_time = full_time[:CONFIG['context_length']]
pred_time = full_time[CONFIG['context_length']:]

ax.plot(context_time, context, label='Context (Input)', linewidth=2.5, color='#95E1D3')
ax.plot(pred_time, true_future, label='True Future', linewidth=2.5, color='#4ECDC4')
ax.plot(pred_time, predicted_values, label='Predicted Future', linewidth=2.5, 
        color='#FF6B6B', linestyle='--', alpha=0.8)

ax.axvline(x=CONFIG['context_length'], color='gray', linestyle=':', alpha=0.5)
ax.text(CONFIG['context_length'], ax.get_ylim()[1], 'Prediction Start', 
        rotation=90, verticalalignment='top', fontsize=10)

ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Value', fontsize=12)
ax.set_title('GPT-2 Time Series Forecasting', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute error metrics
mae = np.abs(predicted_values - true_future).mean()
rmse = np.sqrt(((predicted_values - true_future) ** 2).mean())

print(f"\nForecasting Metrics:")
print(f"  Mean Absolute Error (MAE):  {mae:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse:.4f}")

### Multi-Step Predictions on Multiple Sequences

Test performance across several test sequences.

In [ ]:
def forecast_sequence(model, context, num_steps, discretizer, device):
    """Generate multi-step forecast autoregressively."""
    predictions = []
    current_context = discretizer.encode(context).copy()
    
    model.eval()
    with torch.no_grad():
        for _ in range(num_steps):
            context_tensor = torch.tensor(current_context, dtype=torch.long).unsqueeze(0).to(device)
            logits = model(context_tensor)
            next_token = logits[0, -1, :].argmax().item()
            predictions.append(next_token)
            current_context = np.append(current_context[1:], next_token)
    
    return discretizer.decode(np.array(predictions))


# Evaluate on first 5 test sequences
num_eval = 5
fig, axes = plt.subplots(num_eval, 1, figsize=(14, 3 * num_eval))

maes = []
rmses = []

for i in range(num_eval):
    seq = test_sequences[i]
    context = seq[:CONFIG['context_length']]
    true_future = seq[CONFIG['context_length']:]
    
    # Forecast
    predicted = forecast_sequence(model, context, len(true_future), discretizer, device)
    
    # Metrics
    mae = np.abs(predicted - true_future).mean()
    rmse = np.sqrt(((predicted - true_future) ** 2).mean())
    maes.append(mae)
    rmses.append(rmse)
    
    # Plot
    ax = axes[i]
    full_time = np.arange(len(seq))
    
    ax.plot(full_time[:CONFIG['context_length']], context, 
            linewidth=2, color='#95E1D3', label='Context')
    ax.plot(full_time[CONFIG['context_length']:], true_future, 
            linewidth=2, color='#4ECDC4', label='True')
    ax.plot(full_time[CONFIG['context_length']:], predicted, 
            linewidth=2, color='#FF6B6B', linestyle='--', alpha=0.8, label='Predicted')
    
    ax.axvline(x=CONFIG['context_length'], color='gray', linestyle=':', alpha=0.5)
    ax.set_title(f'Sequence {i+1} | MAE: {mae:.3f} | RMSE: {rmse:.3f}', 
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Time Step', fontsize=10)
    ax.set_ylabel('Value', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAverage Performance:")
print(f"  MAE:  {np.mean(maes):.4f} ± {np.std(maes):.4f}")
print(f"  RMSE: {np.mean(rmses):.4f} ± {np.std(rmses):.4f}")

## 9. Visualize Attention Patterns

### Understanding What the Model Learned

Self-attention reveals which past time steps the model focuses on when making predictions. Let's visualize attention weights from the first layer.

In [ ]:
# Hook to capture attention weights
attention_weights = []

def attention_hook(module, input, output):
    # Store attention weights (before dropout)
    # We'll capture from the attention calculation
    pass

# Simpler approach: manually compute attention for visualization
def get_attention_map(model, context_tensor):
    """Extract attention weights from first layer, first head."""
    model.eval()
    
    with torch.no_grad():
        B, T = context_tensor.size()
        
        # Get embeddings
        tok_emb = model.token_embedding(context_tensor)
        pos = torch.arange(0, T, dtype=torch.long, device=context_tensor.device)
        pos_emb = model.position_embedding(pos)
        x = model.drop(tok_emb + pos_emb)
        
        # Get Q, K from first block
        first_block = model.blocks[0]
        x_norm = first_block.ln_1(x)
        qkv = first_block.attn.c_attn(x_norm)
        q, k, v = qkv.split(model.hparams.n_embd, dim=2)
        
        # Reshape for first head only
        head_dim = model.hparams.n_embd // model.hparams.n_head
        q = q[:, :, :head_dim].squeeze(0)  # (T, head_dim)
        k = k[:, :, :head_dim].squeeze(0)  # (T, head_dim)
        
        # Compute attention
        att = (q @ k.T) / math.sqrt(head_dim)  # (T, T)
        
        # Apply causal mask
        mask = torch.tril(torch.ones(T, T, device=att.device))
        att = att.masked_fill(mask == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
    
    return att.cpu().numpy()


# Get attention for a sample context
sample_context = test_sequences[0, :CONFIG['context_length']]
sample_context_bins = discretizer.encode(sample_context)
context_tensor = torch.tensor(sample_context_bins, dtype=torch.long).unsqueeze(0).to(device)

att_map = get_attention_map(model, context_tensor)

# Plot attention heatmap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Time series
ax1.plot(sample_context, linewidth=2, color='#4ECDC4')
ax1.set_title('Input Time Series', fontsize=14, fontweight='bold')
ax1.set_xlabel('Time Step', fontsize=12)
ax1.set_ylabel('Value', fontsize=12)
ax1.grid(True, alpha=0.3)

# Attention heatmap (last 32 steps for clarity)
plot_len = min(32, CONFIG['context_length'])
im = ax2.imshow(att_map[-plot_len:, -plot_len:], cmap='viridis', aspect='auto')
ax2.set_title('Attention Weights (Layer 1, Head 1)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Key Position (Past)', fontsize=12)
ax2.set_ylabel('Query Position (Current)', fontsize=12)
plt.colorbar(im, ax=ax2, label='Attention Weight')

plt.tight_layout()
plt.show()

print("\nAttention Pattern Insights:")
print("- Diagonal: Strong self-attention (each position attends to itself)")
print("- Lower triangle: Causal mask ensures we only attend to past")
print("- Bright spots: Model learns to focus on relevant past time steps")

## 10. Compare with Baseline: Last Value Persistence

### Is GPT-2 Better Than Just Repeating the Last Value?

A common baseline for time series forecasting is **persistence**: predict the last observed value. Let's compare.

In [ ]:
def baseline_forecast(context, num_steps):
    """Naive baseline: repeat last value."""
    return np.full(num_steps, context[-1])


# Evaluate both on test set
gpt2_maes = []
baseline_maes = []

for seq in test_sequences:
    context = seq[:CONFIG['context_length']]
    true_future = seq[CONFIG['context_length']:]
    
    # GPT-2 forecast
    gpt2_pred = forecast_sequence(model, context, len(true_future), discretizer, device)
    gpt2_mae = np.abs(gpt2_pred - true_future).mean()
    gpt2_maes.append(gpt2_mae)
    
    # Baseline forecast
    baseline_pred = baseline_forecast(context, len(true_future))
    baseline_mae = np.abs(baseline_pred - true_future).mean()
    baseline_maes.append(baseline_mae)

# Results
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(baseline_maes, bins=30, alpha=0.6, label='Baseline (Last Value)', color='#95E1D3')
ax.hist(gpt2_maes, bins=30, alpha=0.6, label='GPT-2', color='#4ECDC4')
ax.axvline(np.mean(baseline_maes), color='#95E1D3', linestyle='--', linewidth=2, 
           label=f'Baseline Mean: {np.mean(baseline_maes):.3f}')
ax.axvline(np.mean(gpt2_maes), color='#4ECDC4', linestyle='--', linewidth=2, 
           label=f'GPT-2 Mean: {np.mean(gpt2_maes):.3f}')

ax.set_xlabel('Mean Absolute Error', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Forecasting Error Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

improvement = (np.mean(baseline_maes) - np.mean(gpt2_maes)) / np.mean(baseline_maes) * 100

print(f"\nModel Comparison:")
print(f"  Baseline MAE: {np.mean(baseline_maes):.4f} ± {np.std(baseline_maes):.4f}")
print(f"  GPT-2 MAE:    {np.mean(gpt2_maes):.4f} ± {np.std(gpt2_maes):.4f}")
print(f"  Improvement:  {improvement:.1f}%")

## 11. Experiment: Effect of Context Length

### How Much History Does the Model Need?

Let's test how prediction quality changes with different context lengths.

In [ ]:
context_lengths = [16, 32, 48, 64]
results = []

for ctx_len in context_lengths:
    maes = []
    
    for seq in test_sequences[:50]:  # Sample 50 sequences for speed
        if len(seq) < ctx_len + 1:
            continue
        
        context = seq[:ctx_len]
        true_future = seq[ctx_len:ctx_len+10]  # Predict next 10 steps
        
        predicted = forecast_sequence(model, context, len(true_future), discretizer, device)
        mae = np.abs(predicted - true_future).mean()
        maes.append(mae)
    
    results.append({
        'context_length': ctx_len,
        'mae_mean': np.mean(maes),
        'mae_std': np.std(maes)
    })

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

ctx_lens = [r['context_length'] for r in results]
mae_means = [r['mae_mean'] for r in results]
mae_stds = [r['mae_std'] for r in results]

ax.errorbar(ctx_lens, mae_means, yerr=mae_stds, marker='o', linewidth=2, 
            markersize=8, capsize=5, color='#4ECDC4')
ax.set_xlabel('Context Length (time steps)', fontsize=12)
ax.set_ylabel('Mean Absolute Error', fontsize=12)
ax.set_title('Effect of Context Length on Forecasting Accuracy', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nContext Length vs. Performance:")
for r in results:
    print(f"  {r['context_length']:2d} steps: MAE = {r['mae_mean']:.4f} ± {r['mae_std']:.4f}")

## 12. Extension: Direct Regression Head (Bonus)

### Alternative Approach: Skip Discretization

Instead of discretizing, we could:
1. Embed continuous values directly (learned projection)
2. Replace classification head with regression (predict real number)

**Pros**: No quantization error, direct optimization of prediction loss  
**Cons**: Loses pretrained weights, different optimization landscape

Here's the key modification:

In [ ]:
class GPT2TimeSeriesRegression(L.LightningModule):
    """GPT-2 with regression head (no discretization)."""
    
    def __init__(
        self,
        context_length=CONFIG['context_length'],
        n_embd=CONFIG['n_embd'],
        n_layer=CONFIG['n_layer'],
        n_head=CONFIG['n_head'],
        dropout=CONFIG['dropout'],
        learning_rate=CONFIG['learning_rate']
    ):
        super().__init__()
        self.save_hyperparameters()
        
        # Value embedding: project scalar to n_embd dimensions
        self.value_embedding = nn.Linear(1, n_embd)
        self.position_embedding = nn.Embedding(context_length, n_embd)
        self.drop = nn.Dropout(dropout)
        
        # Transformer blocks (same as before)
        self.blocks = nn.ModuleList([
            TransformerBlock(n_embd, n_head, context_length, dropout)
            for _ in range(n_layer)
        ])
        
        # Regression head: project back to scalar
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, 1)  # Single continuous output
        
        # Loss
        self.criterion = nn.MSELoss()
    
    def forward(self, x):
        B, T = x.size()
        
        # Embed continuous values
        val_emb = self.value_embedding(x.unsqueeze(-1))  # (B, T, n_embd)
        
        # Position embeddings
        pos = torch.arange(0, T, dtype=torch.long, device=x.device)
        pos_emb = self.position_embedding(pos)
        
        x = self.drop(val_emb + pos_emb)
        
        # Transformer
        for block in self.blocks:
            x = block(x)
        
        x = self.ln_f(x)
        predictions = self.head(x).squeeze(-1)  # (B, T)
        
        return predictions
    
    def training_step(self, batch, batch_idx):
        context, target = batch  # Both are continuous values now
        
        predictions = self(context)  # (B, T)
        prediction = predictions[:, -1]  # Last position
        
        loss = self.criterion(prediction, target)
        self.log('train_loss', loss, prog_bar=True)
        
        return loss
    
    def validation_step(self, batch, batch_idx):
        context, target = batch
        predictions = self(context)
        prediction = predictions[:, -1]
        loss = self.criterion(prediction, target)
        self.log('val_loss', loss, prog_bar=True)
        return loss
    
    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate)


print("\nRegression variant implemented!")
print("Key differences:")
print("  1. Value embedding: Linear(1, n_embd) instead of Embedding(vocab_size, n_embd)")
print("  2. Output head: Linear(n_embd, 1) instead of Linear(n_embd, vocab_size)")
print("  3. Loss: MSE instead of CrossEntropy")
print("\nTo train this variant, you'd need to modify the dataset to return")
print("continuous values instead of discretized bins.")

## 13. Key Takeaways

### What We Learned

1. **Architecture Flexibility**: Transformers are sequence-to-sequence models. Whether the sequence is text tokens or time series values, the self-attention mechanism applies the same way.

2. **Discretization Trade-off**: 
   - **Pros**: Leverage pretrained weights, standard classification setup, stable training
   - **Cons**: Quantization error, limited precision, larger vocabulary

3. **Autoregressive Forecasting**: GPT-2's causal attention ensures predictions only depend on past values, making it naturally suited for time series forecasting.

4. **Context Matters**: Longer context generally improves predictions, but with diminishing returns. The model learns which historical patterns are most informative.

5. **Baselines Are Important**: Simple baselines like persistence provide crucial context for evaluating whether complex models add value.

### When to Use GPT-2 for Time Series

**Good fit**:
- Long-range dependencies (patterns spanning many time steps)
- Irregular sampling or variable-length sequences
- Transfer learning from pretrained language models

**Consider alternatives**:
- Short sequences → simpler models (LSTM, CNN)
- Real-time constraints → computationally cheaper models
- Small datasets → domain-specific architectures

### Next Steps

- Try with **real-world datasets** (stock prices, sensor data, weather)
- Experiment with **multivariate time series** (multiple correlated signals)
- Compare with **specialized time series transformers** (Informer, Autoformer)
- Test **pretrained GPT-2** and fine-tune (transfer learning)